# Test 6: MLP / TF-IDF Baseline (Lower Bound)

**Goal**: Establish a simple non-transformer baseline for the paper comparison table.

| Baseline | Method |
|---|---|
| **LR-TFIDF** | Logistic Regression on TF-IDF character n-grams |
| **MLP-TFIDF** | 2-layer MLP (512â†’256) on the same features |

**Why**: Reviewers expect a lower-bound baseline to show GraphCodeBERT+DFG is non-trivially better.

**Kaggle inputs**: `/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl`

**Outputs**: `test6_baseline_bar.png`, `test6_baseline_results.txt`

**Estimated runtime**: ~25â€“35 min on CPU (no GPU needed)

In [1]:
!pip install scikit-learn matplotlib -q

In [2]:
import os, json, random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    average_precision_score, f1_score,
    classification_report, confusion_matrix
)
random.seed(42)
np.random.seed(42)
print('Imports OK')

Imports OK


In [3]:
# Configuration
DATA_FILE    = '/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl'
OUT_DIR      = '/kaggle/working'
TRAIN_RATIO  = 0.90   # mirrors the main model held-out boundary
SEED         = 42
MAX_FEATURES = 50_000
NGRAM_RANGE  = (1, 3)
ANALYZER     = 'char_wb'  # character n-grams perform best on code
print(f'Data: {DATA_FILE}')
print('Split: 90/10 train/test')

Data: /kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl
Split: 90/10 train/test


In [4]:
# â”€â”€â”€ LOAD DATASET â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Loading dataset...')
texts, labels = [], []
with open(DATA_FILE, encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            entry = json.loads(line)
            texts.append(entry.get('code', ''))
            labels.append(int(entry.get('label', 0)))
        except Exception:
            continue
texts  = np.array(texts,  dtype=object)
labels = np.array(labels, dtype=int)
print(f'Total samples  : {len(texts):,}')
print(f'  Class 0 (safe)     : {(labels==0).sum():,}')
print(f'  Class 1 (malicious): {(labels==1).sum():,}')

Loading dataset...
Total samples  : 199,960
  Class 0 (safe)     : 100,000
  Class 1 (malicious): 99,960


In [5]:
# Train / test split
# Deterministic shuffle then slice; mirrors the main model's 90/10 boundary
idx = np.arange(len(texts))
rng = np.random.default_rng(SEED)
rng.shuffle(idx)
n_train   = int(len(idx) * TRAIN_RATIO)
train_idx = idx[:n_train]
test_idx  = idx[n_train:]
X_train, y_train = texts[train_idx],  labels[train_idx]
X_test,  y_test  = texts[test_idx],   labels[test_idx]
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Test class balance: {y_test.mean():.4f}')

Train: 179,964  |  Test: 19,996
Test class balance: 0.4959


In [6]:
# â”€â”€â”€ BASELINE 1: Logistic Regression + TF-IDF â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Fitting TF-IDF on 160k code samples: ~5-10 min
print('Training LR + TF-IDF  (vectorizer fit ~5-10 min)...')
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer=ANALYZER,
        ngram_range=NGRAM_RANGE,
        max_features=MAX_FEATURES,
        sublinear_tf=True,
        min_df=5,
        strip_accents='unicode',
    )),
    ('clf', LogisticRegression(
        C=1.0, max_iter=1000, solver='saga',
        n_jobs=-1, random_state=SEED,
    ))
])
pipeline_lr.fit(X_train, y_train)
print('  âœ“ LR done')

Training LR + TF-IDF  (vectorizer fit ~5-10 min)...
  âœ“ LR done


In [7]:
# â”€â”€â”€ EVALUATE LR â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
y_pred_lr = pipeline_lr.predict(X_test)
y_prob_lr = pipeline_lr.predict_proba(X_test)[:, 1]
acc_lr    = accuracy_score(y_test, y_pred_lr)
auc_lr    = roc_auc_score(y_test, y_prob_lr)
pr_lr     = average_precision_score(y_test, y_prob_lr)
f1_lr     = f1_score(y_test, y_pred_lr)
fn_lr     = confusion_matrix(y_test, y_pred_lr)[1, 0]
print(f'\nLogistic Regression + TF-IDF')
print(f'  Accuracy : {acc_lr:.4%}')
print(f'  ROC-AUC  : {auc_lr:.4f}')
print(f'  PR-AUC   : {pr_lr:.4f}')
print(f'  F1       : {f1_lr:.4f}')
print(f'  FN (missed malware): {fn_lr:,}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['safe','malicious']))


Logistic Regression + TF-IDF
  Accuracy : 84.4969%
  ROC-AUC  : 0.9277
  PR-AUC   : 0.9227
  F1       : 0.8427
  FN (missed malware): 1,615

              precision    recall  f1-score   support

        safe       0.84      0.85      0.85     10079
   malicious       0.85      0.84      0.84      9917

    accuracy                           0.84     19996
   macro avg       0.85      0.84      0.84     19996
weighted avg       0.85      0.84      0.84     19996



In [8]:
# â”€â”€â”€ BASELINE 2: MLP + TF-IDF â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Reuse the already-fitted vectorizer from LR pipeline to save time
print('Transforming features (reusing fitted TF-IDF)...')
vectorizer    = pipeline_lr.named_steps['tfidf']
X_train_vec   = vectorizer.transform(X_train)
X_test_vec    = vectorizer.transform(X_test)
print(f'  Feature matrix: {X_train_vec.shape}')
print('Training MLP 512->256  (~10-15 min on CPU)...')
mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation='relu',
    max_iter=30,
    batch_size=256,
    learning_rate_init=1e-3,
    early_stopping=True,
    validation_fraction=0.05,
    n_iter_no_change=5,
    verbose=True,
    random_state=SEED,
)
mlp.fit(X_train_vec, y_train)
print('  âœ“ MLP done')

Transforming features (reusing fitted TF-IDF)...
  Feature matrix: (179964, 50000)
Training MLP 512->256  (~10-15 min on CPU)...
Iteration 1, loss = 0.35432390
Validation score: 0.858206
Iteration 2, loss = 0.23697076
Validation score: 0.855540
Iteration 3, loss = 0.13898362
Validation score: 0.854095
Iteration 4, loss = 0.07964919
Validation score: 0.850539
Iteration 5, loss = 0.06105726
Validation score: 0.848316
Iteration 6, loss = 0.05022310
Validation score: 0.851539
Iteration 7, loss = 0.04572894
Validation score: 0.853539
Validation score did not improve more than tol=0.000100 for 5 consecutive epochs. Stopping.
  âœ“ MLP done


In [9]:
# â”€â”€â”€ EVALUATE MLP â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
y_pred_mlp = mlp.predict(X_test_vec)
y_prob_mlp = mlp.predict_proba(X_test_vec)[:, 1]
acc_mlp    = accuracy_score(y_test, y_pred_mlp)
auc_mlp    = roc_auc_score(y_test, y_prob_mlp)
pr_mlp     = average_precision_score(y_test, y_prob_mlp)
f1_mlp     = f1_score(y_test, y_pred_mlp)
fn_mlp     = confusion_matrix(y_test, y_pred_mlp)[1, 0]
print(f'\nMLP (512-256) + TF-IDF')
print(f'  Accuracy : {acc_mlp:.4%}')
print(f'  ROC-AUC  : {auc_mlp:.4f}')
print(f'  PR-AUC   : {pr_mlp:.4f}')
print(f'  F1       : {f1_mlp:.4f}')
print(f'  FN (missed malware): {fn_mlp:,}')
print()
print(classification_report(y_test, y_pred_mlp, target_names=['safe','malicious']))


MLP (512-256) + TF-IDF
  Accuracy : 85.5771%
  ROC-AUC  : 0.9393
  PR-AUC   : 0.9374
  F1       : 0.8567
  FN (missed malware): 1,298

              precision    recall  f1-score   support

        safe       0.87      0.84      0.85     10079
   malicious       0.84      0.87      0.86      9917

    accuracy                           0.86     19996
   macro avg       0.86      0.86      0.86     19996
weighted avg       0.86      0.86      0.86     19996



In [10]:
# Full paper comparison table
# Reference numbers from earlier tests (updated final transformer results)
REF = [
    ('LR + TF-IDF',           acc_lr,   auc_lr,  pr_lr,  f1_lr,   fn_lr),
    ('MLP + TF-IDF',          acc_mlp,  auc_mlp, pr_mlp, f1_mlp,  fn_mlp),
    ('CodeBERT (text-only)',  0.8848,   0.9610,  0.9616, 0.8848,  1072),
    ('UniXcoder (text-only)', 0.892829, 0.9652,  0.9657, 0.8928,  1051),
    ('GCB + DFG (ours)',      0.887077, 0.9616,  0.9622, 0.8871,  1184),
]
print(f'{"Method":<30} {"Acc":>8} {"ROC-AUC":>9} {"PR-AUC":>8} {"F1":>7} {"FN":>6}')
print('-'*75)
for nm, acc, auc_, pr, f1, fn in REF:
    fn_s = f'{int(fn):,}' if fn is not None else '    -'
    print(f'{nm:<30} {acc:>8.4%} {auc_:>9.4f} {pr:>8.4f} {f1:>7.4f} {fn_s:>6}')

Method                              Acc   ROC-AUC   PR-AUC      F1     FN
---------------------------------------------------------------------------
LR + TF-IDF                    84.4969%    0.9277   0.9227  0.8427  1,615
MLP + TF-IDF                   85.5771%    0.9393   0.9374  0.8567  1,298
CodeBERT (text-only)           88.4800%    0.9610   0.9616  0.8848  1,072
UniXcoder (text-only)          89.2829%    0.9652   0.9657  0.8928  1,051
GCB + DFG (ours)               88.7077%    0.9616   0.9622  0.8871  1,184


In [11]:
# Bar chart
labels_plot = ['LR+\nTF-IDF', 'MLP+\nTF-IDF', 'CodeBERT', 'UniXcoder', 'GCB+DFG']
vals_acc    = [acc_lr, acc_mlp, 0.8848, 0.892829, 0.887077]
vals_auc    = [auc_lr, auc_mlp, 0.9610, 0.9652, 0.9616]
x = np.arange(len(labels_plot))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w/2, vals_acc, w, label='Accuracy', color='#4C72B0', alpha=0.88)
b2 = ax.bar(x + w/2, vals_auc, w, label='ROC-AUC', color='#55A868', alpha=0.88)
for bar in list(b1) + list(b2):
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)
ax.set_ylim(0.82, 1.00)
ax.set_xticks(x)
ax.set_xticklabels(labels_plot, fontsize=10)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Accuracy & ROC-AUC: TF-IDF Baselines vs Final Transformer Models', fontsize=14)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
ax.axvspan(-0.5, 1.5, alpha=0.07, color='red')
ax.text(0.5, 1.02, 'Baselines', ha='center', fontsize=10, color='#880000',
        transform=ax.get_xaxis_transform())
out_img = '/kaggle/working/test6_baseline_bar.png'
fig.savefig(out_img, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved -> {out_img}')

Saved -> /kaggle/working/test6_baseline_bar.png


In [12]:
# Save results txt
out_txt = '/kaggle/working/test6_baseline_results.txt'
with open(out_txt, 'w') as fh:
    fh.write('Test 6: MLP / TF-IDF Baseline Results\n')
    fh.write('=' * 60 + '\n')
    for nm, acc, auc_, pr, f1, fn in REF:
        fn_s = f'{int(fn):,}' if fn is not None else '-'
        fh.write(f'\n{nm}\n')
        fh.write(f'  Accuracy : {acc:.4%}\n')
        fh.write(f'  ROC-AUC  : {auc_:.4f}\n')
        fh.write(f'  PR-AUC   : {pr:.4f}\n')
        fh.write(f'  F1       : {f1:.4f}\n')
        fh.write(f'  FN       : {fn_s}\n')
print(f'Saved -> {out_txt}')
print(f'Saved -> {out_img}')

Saved -> /kaggle/working/test6_baseline_results.txt
Saved -> /kaggle/working/test6_baseline_bar.png
